# nb24 - Free squeeze: seed + architecture ensembling

Before spending GPU on larger windows, extract the free gains already latent in the nb19 predictions: averaging the 5 per-seed predictions (seed ensemble) and averaging across architectures (model ensemble). Both need no retraining - just the saved per-cluster CSVs, which share the same fixed test split (split seed 0), so predictions align row-for-row across seeds and models on min-bias.

In [1]:
import sys, pathlib, os
import numpy as np, pandas as pd
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import resolution
PRED = REPO / 'reports' / 'predictions'
def load(model, dataset='minbias'):
    return pd.read_csv(PRED / f'{dataset}__{model}.csv')
MODELS = ['GateHuber', 'Spacetime', 'EfnResidual', 'PairT', 'MeanResidual', 'MeanDirect']
print('available:', [m for m in MODELS if (PRED / f'minbias__{m}.csv').exists()])

available: ['GateHuber', 'Spacetime', 'EfnResidual', 'PairT', 'MeanResidual', 'MeanDirect']


## Alignment check
Each model CSV has 5 seeds x the same test clusters. Confirm every seed block has identical length and identical true_energy ordering, so we can average predictions by position within a seed block.

In [2]:
def seed_matrix(df):
    seeds = sorted(df['seed'].unique())
    blocks = [df[df.seed == s].reset_index(drop=True) for s in seeds]
    n = min(len(b) for b in blocks)
    te = np.stack([b['true_energy'].to_numpy()[:n] for b in blocks])
    ok = np.allclose(te, te[0])
    P = np.stack([b['pred_energy'].to_numpy()[:n] for b in blocks])
    return te[0], P, ok
gh = load('GateHuber'); true_e, Pgh, ok = seed_matrix(gh)
print('GateHuber: seeds', gh['seed'].nunique(), 'clusters', len(true_e), 'aligned across seeds:', ok)

GateHuber: seeds 5 clusters 12227 aligned across seeds: True


## 1) Per-seed vs seed-ensemble (GateHuber)
Seed ensemble = mean of the 5 seed predictions per cluster.

In [3]:
per_seed = [resolution(Pgh[i], true_e)['sigma_eff'] for i in range(Pgh.shape[0])]
ens = resolution(Pgh.mean(0), true_e)['sigma_eff']
ens_log = resolution(np.exp(np.log(Pgh).mean(0)), true_e)['sigma_eff']
print('GateHuber per-seed sigma_eff:', [round(s,4) for s in per_seed], '-> mean', round(np.mean(per_seed),4))
print('seed-ensemble (arith mean):', round(ens,4))
print('seed-ensemble (geo mean)  :', round(ens_log,4))

GateHuber per-seed sigma_eff: [0.0486, 0.0493, 0.047, 0.0487, 0.0491] -> mean 0.0485
seed-ensemble (arith mean): 0.0463
seed-ensemble (geo mean)  : 0.0463


## 2) Cross-architecture ensemble
Average the seed-ensembled prediction of several architectures (they must share the aligned test set).

In [4]:
def seed_ens_pred(model):
    _, P, ok = seed_matrix(load(model))
    return P.mean(0), ok
avail = [m for m in MODELS if (PRED / f'minbias__{m}.csv').exists()]
preds = {}; base_te = true_e
for m in avail:
    p, ok = seed_ens_pred(m)
    if len(p) == len(base_te) and ok: preds[m] = p
for combo in [['GateHuber'], ['GateHuber','Spacetime'], ['GateHuber','Spacetime','EfnResidual'],
              ['GateHuber','Spacetime','EfnResidual','PairT'], list(preds.keys())]:
    combo = [m for m in combo if m in preds]
    if not combo: continue
    stack = np.stack([preds[m] for m in combo])
    s = resolution(stack.mean(0), base_te)['sigma_eff']
    print(f'{"+".join(combo):55s} sigma_eff {s:.4f}')

GateHuber                                               sigma_eff 0.0463
GateHuber+Spacetime                                     sigma_eff 0.0496
GateHuber+Spacetime+EfnResidual                         sigma_eff 0.0515
GateHuber+Spacetime+EfnResidual+PairT                   sigma_eff 0.0536
GateHuber+Spacetime+EfnResidual+PairT+MeanResidual+MeanDirect sigma_eff 0.0563


## Summary
Free gains from ensembling vs the nb19 single-seed GateHuber (0.0485). These stack on top of any window/architecture improvement and cost no extra training.

In [5]:
print('nb19 single-seed GateHuber (kNN-25): 0.0485')
print('best ensemble here            :', round(min(
    resolution(np.stack([preds[m] for m in preds]).mean(0), base_te)['sigma_eff'],
    resolution(Pgh.mean(0), true_e)['sigma_eff']), 4))

nb19 single-seed GateHuber (kNN-25): 0.0485
best ensemble here            : 0.0463
